In [ ]:
import os
from dotenv import load_dotenv
from supabase import create_client
import pandas as pd
from datetime import datetime, timedelta,date


# Load env
load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print(SUPABASE_URL)

response = supabase.rpc("get_tanggal_terakhir_per_tipe").execute()
data = response.data

# Ambil tanggal terakhir dari supabase
tanggal_terakhir_str = data[2]["tanggal_terakhir"]

# Ubah ke tipe date (bukan string)
tanggal_terakhir = pd.to_datetime(tanggal_terakhir_str).date()

# Tambah 1 hari
next_date = tanggal_terakhir + pd.Timedelta(days=1)

# Ambil hari ini sebagai date
today = pd.Timestamp.today().date()

print("Tanggal terakhir :", tanggal_terakhir)
print("Next date        :", next_date)
print("Hari ini         :", today)

https://extlxiwpcbzqaalpopqn.supabase.co
Tanggal terakhir : 2026-02-25
Next date        : 2026-02-26
Hari ini         : 2026-02-26


In [1]:
import requests
import pandas as pd
from io import BytesIO
import os
import datetime

def get_bapanas_dataframe(tanggal, level_harga_id=3, province_id=15, filter_beras=False):
    """
    Mengambil data dari API BAPANAS dan langsung mengembalikan DataFrame
    tanpa menyimpan file Excel.

    Args:
        tanggal: datetime.date object atau string format 'YYYY-MM-DD'
        level_harga_id: int (3=konsumen, 1=produsen, dst)
        province_id: int (15=Jawa Timur, default)
        filter_beras: bool (True=hanya Beras Premium & Medium, False=semua data)

    Returns:
        tuple: (success: bool, dataframe: pd.DataFrame, message: str)

    Example:
        >>> import datetime
        >>> success, df, msg = get_bapanas_dataframe(datetime.date(2024, 1, 15), filter_beras=True)
        >>> if success:
        >>>     print(df.head())
    """
    # Konversi string ke datetime.date jika perlu
    if isinstance(tanggal, str):
        tanggal = datetime.datetime.strptime(tanggal, '%Y-%m-%d').date()

    # Format tanggal untuk API
    formatted = tanggal.strftime("%d/%m/%Y")
    period_param = f"{formatted}%20-%20{formatted}"

    # URL API BAPANAS
    url = f"https://api-panelhargav2.badanpangan.go.id/harga-pangan-table-province/export?province_id={province_id}&period_date={period_param}&level_harga_id={level_harga_id}"

    try:
        response = requests.get(url, timeout=60)

        if response.status_code == 200:
            # Baca Excel langsung dari memory tanpa menyimpan ke file
            df = pd.read_excel(BytesIO(response.content))

            if df.empty:
                return False, df, "Data kosong atau tidak tersedia untuk tanggal tersebut"

            # Filter hanya Beras Premium dan Beras Medium jika diminta
            if filter_beras:
                # Cari kolom identifikasi (No, Kota/Kabupaten, dll)
                kolom_identitas = []
                for col in df.columns:
                    col_lower = col.lower()
                    if any(keyword in col_lower for keyword in ['no', 'kota', 'kabupaten', 'wilayah', 'daerah', 'provinsi']):
                        kolom_identitas.append(col)

                # Cari kolom Beras Premium dan Beras Medium
                kolom_beras = []
                for col in df.columns:
                    if 'Beras Premium' in col or 'Beras Medium' in col:
                        kolom_beras.append(col)

                if not kolom_beras:
                    return False, pd.DataFrame(), "Kolom Beras Premium atau Beras Medium tidak ditemukan"

                # Pilih hanya kolom identitas + kolom beras
                kolom_terpilih = kolom_identitas + kolom_beras
                df = df[kolom_terpilih]

                if df.empty:
                    return False, df, "Data kosong setelah filter"

                return True, df, f"Berhasil mengambil {len(df)} baris data (hanya kolom {', '.join(kolom_beras)})"

            return True, df, f"Berhasil mengambil {len(df)} baris data"
        else:
            return False, pd.DataFrame(), f"HTTP Error {response.status_code}"

    except requests.exceptions.Timeout:
        return False, pd.DataFrame(), "Request timeout - API tidak merespons"
    except requests.exceptions.RequestException as e:
        return False, pd.DataFrame(), f"Request error: {str(e)}"
    except Exception as e:
        return False, pd.DataFrame(), f"Error: {str(e)}"


def get_bapanas_konsumen_produsen(tanggal, province_id=15, filter_beras=False):
    """
    Mengambil data konsumen DAN produsen sekaligus dari API BAPANAS.

    Args:
        tanggal: datetime.date object atau string format 'YYYY-MM-DD'
        province_id: int (15=Jawa Timur, default)
        filter_beras: bool (True=hanya Beras Premium & Medium, False=semua data)

    Returns:
        tuple: (success: bool, df_konsumen: pd.DataFrame, df_produsen: pd.DataFrame, message: str)

    Example:
        >>> import datetime
        >>> success, df_konsumen, df_produsen, msg = get_bapanas_konsumen_produsen(datetime.date.today(), filter_beras=True)
        >>> if success:
        >>>     print("Konsumen:", len(df_konsumen), "baris")
        >>>     print("Produsen:", len(df_produsen), "baris")
    """
    # Ambil data konsumen (level_harga_id=3)
    success_konsumen, df_konsumen, msg_konsumen = get_bapanas_dataframe(
        tanggal, level_harga_id=3, province_id=province_id, filter_beras=filter_beras
    )

    # Ambil data produsen (level_harga_id=1)
    success_produsen, df_produsen, msg_produsen = get_bapanas_dataframe(
        tanggal, level_harga_id=1, province_id=province_id, filter_beras=filter_beras
    )

    # Evaluasi hasil
    if success_konsumen and success_produsen:
        message = f"✅ Berhasil mengambil keduanya.\nKonsumen: {len(df_konsumen)} baris\nProdusen: {len(df_produsen)} baris"
        return True, df_konsumen, df_produsen, message
    elif success_konsumen:
        message = f"⚠️ Hanya konsumen berhasil ({len(df_konsumen)} baris). Produsen gagal: {msg_produsen}"
        return True, df_konsumen, pd.DataFrame(), message
    elif success_produsen:
        message = f"⚠️ Hanya produsen berhasil ({len(df_produsen)} baris). Konsumen gagal: {msg_konsumen}"
        return True, pd.DataFrame(), df_produsen, message
    else:
        message = f"❌ Keduanya gagal.\nKonsumen: {msg_konsumen}\nProdusen: {msg_produsen}"
        return False, pd.DataFrame(), pd.DataFrame(), message


def get_bapanas_periode(start_date, end_date, level_harga_id=3, province_id=15, filter_beras=False):
    """
    Mengambil data BAPANAS untuk rentang tanggal dan menggabungkannya
    menjadi satu DataFrame.

    Args:
        start_date: datetime.date object atau string 'YYYY-MM-DD'
        end_date: datetime.date object atau string 'YYYY-MM-DD'
        level_harga_id: int (3=konsumen, 1=produsen, dst)
        province_id: int (15=Jawa Timur, default)
        filter_beras: bool (True=hanya Beras Premium & Medium, False=semua data)

    Returns:
        tuple: (success: bool, dataframe: pd.DataFrame, message: str)

    Example:
        >>> success, df, msg = get_bapanas_periode('2024-01-01', '2024-01-07', filter_beras=True)
        >>> if success:
        >>>     print(f"Total data: {len(df)} baris")
    """
    # Konversi string ke datetime.date jika perlu
    if isinstance(start_date, str):
        start_date = datetime.datetime.strptime(start_date, '%Y-%m-%d').date()
    if isinstance(end_date, str):
        end_date = datetime.datetime.strptime(end_date, '%Y-%m-%d').date()

    if start_date > end_date:
        return False, pd.DataFrame(), "Tanggal mulai tidak boleh lebih besar dari tanggal akhir"

    all_data = []
    total_days = (end_date - start_date).days + 1
    failed_dates = []

    for i in range(total_days):
        current_date = start_date + datetime.timedelta(days=i)
        success, df, msg = get_bapanas_dataframe(current_date, level_harga_id, province_id, filter_beras)

        if success and not df.empty:
            # Tambahkan kolom tanggal untuk tracking
            df['tanggal_scrape'] = current_date
            all_data.append(df)
        else:
            failed_dates.append(current_date.strftime('%Y-%m-%d'))

    if not all_data:
        return False, pd.DataFrame(), "Tidak ada data berhasil diambil"

    # Gabungkan semua dataframe
    result_df = pd.concat(all_data, ignore_index=True)

    filter_msg = " (Beras Premium & Medium)" if filter_beras else ""
    message = f"Berhasil mengambil {len(all_data)} dari {total_days} hari. Total {len(result_df)} baris data{filter_msg}."
    if failed_dates:
        message += f"\nGagal: {', '.join(failed_dates[:5])}"
        if len(failed_dates) > 5:
            message += f" ... (+{len(failed_dates)-5} lainnya)"

    return True, result_df, message


def get_bapanas_periode_konsumen_produsen(start_date, end_date, province_id=15, filter_beras=False):
    """
    Mengambil data konsumen DAN produsen untuk rentang tanggal.

    Args:
        start_date: datetime.date object atau string 'YYYY-MM-DD'
        end_date: datetime.date object atau string 'YYYY-MM-DD'
        province_id: int (15=Jawa Timur, default)
        filter_beras: bool (True=hanya Beras Premium & Medium, False=semua data)

    Returns:
        tuple: (success: bool, df_konsumen: pd.DataFrame, df_produsen: pd.DataFrame, message: str)

    Example:
        >>> success, df_k, df_p, msg = get_bapanas_periode_konsumen_produsen('2024-01-01', '2024-01-07', filter_beras=True)
        >>> if success:
        >>>     print("Konsumen:", len(df_k), "baris")
        >>>     print("Produsen:", len(df_p), "baris")
    """
    # Ambil data konsumen untuk periode
    success_konsumen, df_konsumen, msg_konsumen = get_bapanas_periode(
        start_date, end_date, level_harga_id=3, province_id=province_id, filter_beras=filter_beras
    )

    # Ambil data produsen untuk periode
    success_produsen, df_produsen, msg_produsen = get_bapanas_periode(
        start_date, end_date, level_harga_id=1, province_id=province_id, filter_beras=filter_beras
    )

    # Evaluasi hasil
    if success_konsumen and success_produsen:
        message = f"✅ Berhasil mengambil keduanya.\nKonsumen: {len(df_konsumen)} baris\nProdusen: {len(df_produsen)} baris"
        return True, df_konsumen, df_produsen, message
    elif success_konsumen:
        message = f"⚠️ Hanya konsumen berhasil ({len(df_konsumen)} baris).\nProdusen: {msg_produsen}"
        return True, df_konsumen, pd.DataFrame(), message
    elif success_produsen:
        message = f"⚠️ Hanya produsen berhasil ({len(df_produsen)} baris).\nKonsumen: {msg_konsumen}"
        return True, pd.DataFrame(), df_produsen, message
    else:
        message = f"❌ Keduanya gagal.\nKonsumen: {msg_konsumen}\nProdusen: {msg_produsen}"
        return False, pd.DataFrame(), pd.DataFrame(), message

def deduplicate_columns(df):
    """Rename duplicate columns by appending _1, _2, etc."""
    cols = pd.Series(df.columns)
    for dup in cols[cols.duplicated()].unique():
        mask = cols == dup
        cols[mask] = [f"{dup}_{i}" if i != 0 else dup for i, _ in enumerate(mask[mask].index)]
    df.columns = cols
    return df




print("Selesai ✅")

Selesai ✅


In [61]:
import pandas as pd
import numpy as np

def clean_bapanas(df, tipe_harga, tanggal):

    # Filter tipe
    df = df[df["tipe"] == tipe_harga]

    if df.empty:
        return pd.DataFrame()

    df = df.copy()
    df["tanggal"] = tanggal

    # Tentukan kolom & tipe_harga_id
    if tipe_harga == "produsen":
        kolom_beras = ["Beras Medium Penggilingan", "Beras Premium Penggilingan"]
        df["tipe_harga_id"] = 3

    elif tipe_harga == "konsumen":
        kolom_beras = ["Beras Medium", "Beras Premium"]
        df["tipe_harga_id"] = 2

    else:
        raise ValueError(f"Tipe tidak dikenali: {tipe_harga}")

    kolom_pakai = kolom_beras + ["No", "Kota/Kabupaten", "tanggal", "tipe_harga_id"]

    df = df[kolom_pakai]

    # Normalisasi nama kolom
    df.columns = (
        df.columns
            .str.strip()
            .str.lower()
            .str.replace(r"\s+", "_", regex=True)
    )

    df = df.rename(columns={"kota/kabupaten": "kota"})

    # Rename kolom beras lebih simpel
    rename_map = {
        col: "beras_medium" for col in df.columns if "medium" in col
    }
    rename_map.update({
        col: "beras_premium" for col in df.columns if "premium" in col
    })

    df = df.rename(columns=rename_map)

    # Ambil hanya No numerik
    df["no"] = pd.to_numeric(df["no"], errors="coerce")
    df = df[df["no"].notna()].copy()
    df["no"] = df["no"].astype(int)

    kolom_komoditas = ["beras_medium", "beras_premium"]

    for col in kolom_komoditas:
        df[col] = (
            pd.to_numeric(df[col], errors="coerce")
            .replace(0, pd.NA)
        )

    # Hitung rerata
    rerata = df[kolom_komoditas].mean()

    # Tambah baris Jawa Timur
    new_row = {
        "no": df["no"].max() + 1,
        "kota": "Jawa Timur",
        "tanggal": df["tanggal"].iloc[0],
        "tipe_harga_id": df["tipe_harga_id"].iloc[0]
    }

    for col in kolom_komoditas:
        new_row[col] = rerata[col]

    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

    # Pilih kolom akhir
    df = df[["kota", "tanggal", "tipe_harga_id"] + kolom_komoditas]

    # Melt
    df = df.melt(
        id_vars=["kota", "tanggal", "tipe_harga_id"],
        value_vars=kolom_komoditas,
        var_name="variant",
        value_name="harga"
    )

    df = encoding(df)
    df["tanggal"] = pd.to_datetime(df["tanggal"]).dt.strftime("%Y-%m-%d")
    df["harga"] = df["harga"].round().astype("Int64")

    return df

def encoding(df):
    kota_jatim = [
        {"kode_kab_kota": 3501, "nama_kab_kota": "Kab. Pacitan"},
        {"kode_kab_kota": 3502, "nama_kab_kota": "Kab. Ponorogo"},
        {"kode_kab_kota": 3503, "nama_kab_kota": "Kab. Trenggalek"},
        {"kode_kab_kota": 3504, "nama_kab_kota": "Kab. Tulungagung"},
        {"kode_kab_kota": 3505, "nama_kab_kota": "Kab. Blitar"},
        {"kode_kab_kota": 3506, "nama_kab_kota": "Kab. Kediri"},
        {"kode_kab_kota": 3507, "nama_kab_kota": "Kab. Malang"},
        {"kode_kab_kota": 3508, "nama_kab_kota": "Kab. Lumajang"},
        {"kode_kab_kota": 3509, "nama_kab_kota": "Kab. Jember"},
        {"kode_kab_kota": 3510, "nama_kab_kota": "Kab. Banyuwangi"},
        {"kode_kab_kota": 3511, "nama_kab_kota": "Kab. Bondowoso"},
        {"kode_kab_kota": 3512, "nama_kab_kota": "Kab. Situbondo"},
        {"kode_kab_kota": 3513, "nama_kab_kota": "Kab. Probolinggo"},
        {"kode_kab_kota": 3514, "nama_kab_kota": "Kab. Pasuruan"},
        {"kode_kab_kota": 3515, "nama_kab_kota": "Kab. Sidoarjo"},
        {"kode_kab_kota": 3516, "nama_kab_kota": "Kab. Mojokerto"},
        {"kode_kab_kota": 3517, "nama_kab_kota": "Kab. Jombang"},
        {"kode_kab_kota": 3518, "nama_kab_kota": "Kab. Nganjuk"},
        {"kode_kab_kota": 3519, "nama_kab_kota": "Kab. Madiun"},
        {"kode_kab_kota": 3520, "nama_kab_kota": "Kab. Magetan"},
        {"kode_kab_kota": 3521, "nama_kab_kota": "Kab. Ngawi"},
        {"kode_kab_kota": 3522, "nama_kab_kota": "Kab. Bojonegoro"},
        {"kode_kab_kota": 3523, "nama_kab_kota": "Kab. Tuban"},
        {"kode_kab_kota": 3524, "nama_kab_kota": "Kab. Lamongan"},
        {"kode_kab_kota": 3525, "nama_kab_kota": "Kab. Gresik"},
        {"kode_kab_kota": 3526, "nama_kab_kota": "Kab. Bangkalan"},
        {"kode_kab_kota": 3527, "nama_kab_kota": "Kab. Sampang"},
        {"kode_kab_kota": 3528, "nama_kab_kota": "Kab. Pamekasan"},
        {"kode_kab_kota": 3529, "nama_kab_kota": "Kab. Sumenep"},
        {"kode_kab_kota": 3571, "nama_kab_kota": "Kota Kediri"},
        {"kode_kab_kota": 3572, "nama_kab_kota": "Kota Blitar"},
        {"kode_kab_kota": 3573, "nama_kab_kota": "Kota Malang"},
        {"kode_kab_kota": 3574, "nama_kab_kota": "Kota Probolinggo"},
        {"kode_kab_kota": 3575, "nama_kab_kota": "Kota Pasuruan"},
        {"kode_kab_kota": 3576, "nama_kab_kota": "Kota Mojokerto"},
        {"kode_kab_kota": 3577, "nama_kab_kota": "Kota Madiun"},
        {"kode_kab_kota": 3578, "nama_kab_kota": "Kota Surabaya"},
        {"kode_kab_kota": 3579, "nama_kab_kota": "Kota Batu"},
        {"kode_kab_kota": 1, "nama_kab_kota": "Jawa Timur"}
    ]

    mapping_kota = {
        item["nama_kab_kota"]: item["kode_kab_kota"]
        for item in kota_jatim
    }
    df["kode_kab_kota"] = df["kota"].map(mapping_kota)

    if df["kode_kab_kota"].isna().any():
        missing = df[df["kode_kab_kota"].isna()]["kota"].unique()
        raise ValueError(f"Kota tidak ter-mapping: {missing}")
        # mapping variant
    df['variant_clean'] = df['variant'].str.lower()

    conditions = [  
        df['variant_clean'] == "beras_medium",
        df['variant_clean'] == "beras_premium",
        df['harga'].isna()
    ]

    choices = [1, 2, np.nan]


    df['variant_id'] = np.select(conditions, choices, default=99)


    df = remove_null(df)
    
    return df[["kode_kab_kota", "tanggal", "variant_id", "harga", "tipe_harga_id"]]

def remove_null(df):
    df = df.loc[
        pd.to_numeric(df["harga"].replace("-", None), errors="coerce")
        .pipe(lambda s: (s.notna()) & (s != 0))
    ]
    return df.reset_index(drop=True)


def clean_sp2kp(df):
    df['tipe_harga_id'] = 1
    df['variant'] = (
        df['variant']
            .str.strip()
            .str.lower()
            .str.replace(r"\s+", "_", regex=True)
    )
    df['tanggal'] = df['date']
    df["harga"] = df["harga"]

    df = encoding(df)

    df = remove_null(df)

    return df[["kode_kab_kota", "tanggal", "variant_id", "harga", 'tipe_harga_id']]



In [62]:
import pandas as pd
import time

# ===============================
# PARAMETER
# ===============================
start_date = next_date
end_date   = today

MAX_RETRY = 3
DELAY_BETWEEN_REQUEST = 2  # detik

# ===============================
# STORAGE
# ===============================
all_konsumen = []
all_produsen = []
error_log = []

# ===============================
# LOOP TANGGAL
# ===============================
for tanggal in pd.date_range(start=start_date, end=end_date):

    tanggal_str = tanggal.strftime("%Y-%m-%d")
    print("\n==============================")
    print("Proses:", tanggal_str)

    success = False

    # ===============================
    # RETRY LOGIC
    # ===============================
    for attempt in range(1, MAX_RETRY + 1):

        print(f"Percobaan ke-{attempt}")

        try:
            status, df_konsumen, df_produsen, message = get_bapanas_konsumen_produsen(
                tanggal_str, 
                filter_beras=False
            )

            if status:
                success = True
                break
            else:
                print("Gagal:", message)

        except Exception as e:
            print("Error:", e)
            message = str(e)

        time.sleep(3)  # delay sebelum retry

    # ===============================
    # JIKA SEMUA RETRY GAGAL
    # ===============================
    if not success:
        print("❌ Gagal total:", tanggal_str)

        error_log.append({
            "tanggal": tanggal_str,
            "error": message
        })

        continue

    inserted = False

    # ===============================
    # PROSES KONSUMEN
    # ===============================
    if df_konsumen is not None and not df_konsumen.empty:
        df_konsumen = deduplicate_columns(df_konsumen)
        df_konsumen["tipe"] = "konsumen"
        df_konsumen = clean_bapanas(df_konsumen, "konsumen", tanggal_str)
        all_konsumen.append(df_konsumen)
        inserted = True

    # ===============================
    # PROSES PRODUSEN
    # ===============================
    if df_produsen is not None and not df_produsen.empty:
        df_produsen = deduplicate_columns(df_produsen)
        df_produsen["tipe"] = "produsen"
        df_produsen = clean_bapanas(df_produsen, "produsen", tanggal_str)
        all_produsen.append(df_produsen)
        inserted = True

    if not inserted:
        print("⚠ Data kosong:", tanggal_str)
        error_log.append({
            "tanggal": tanggal_str,
            "error": "Data kosong"
        })

    # delay sebelum lanjut tanggal berikutnya
    time.sleep(DELAY_BETWEEN_REQUEST)

# ===============================
# CONCAT FINAL DATA
# ===============================
df_all_konsumen = pd.concat(all_konsumen, ignore_index=True) if all_konsumen else pd.DataFrame()
df_all_produsen = pd.concat(all_produsen, ignore_index=True) if all_produsen else pd.DataFrame()

# ===============================
# CEK TANGGAL YANG BENAR-BENAR HILANG
# ===============================
expected_dates = set(pd.date_range(start_date, end_date).strftime("%Y-%m-%d"))

konsumen_dates = set(df_all_konsumen["tanggal"].astype(str)) if not df_all_konsumen.empty else set()
produsen_dates = set(df_all_produsen["tanggal"].astype(str)) if not df_all_produsen.empty else set()

missing_konsumen = expected_dates - konsumen_dates
missing_produsen = expected_dates - produsen_dates

print("\n==============================")
print("Missing Konsumen:", missing_konsumen)
print("Missing Produsen:", missing_produsen)

print("\nSelesai.")


Proses: 2026-02-21
Percobaan ke-1

Proses: 2026-02-22
Percobaan ke-1

Proses: 2026-02-23
Percobaan ke-1

Proses: 2026-02-24
Percobaan ke-1

Proses: 2026-02-25
Percobaan ke-1

Missing Konsumen: set()
Missing Produsen: set()

Selesai.


In [63]:
import requests
import pandas as pd
from datetime import date
from dateutil.relativedelta import relativedelta
import time

# URL API
url = "https://api-sp2kp.kemendag.go.id/report/api/average-price/export-area-daily-json"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Origin": "https://sp2kp.kemendag.go.id",
    "Referer": "https://sp2kp.kemendag.go.id/",
    "Accept": "application/json"
}

kota_jatim = [
    {"kode_kab_kota": 3501, "nama_kab_kota": "Kab. Pacitan"},
    {"kode_kab_kota": 3502, "nama_kab_kota": "Kab. Ponorogo"},
    {"kode_kab_kota": 3503, "nama_kab_kota": "Kab. Trenggalek"},
    {"kode_kab_kota": 3504, "nama_kab_kota": "Kab. Tulungagung"},
    {"kode_kab_kota": 3505, "nama_kab_kota": "Kab. Blitar"},
    {"kode_kab_kota": 3506, "nama_kab_kota": "Kab. Kediri"},
    {"kode_kab_kota": 3507, "nama_kab_kota": "Kab. Malang"},
    {"kode_kab_kota": 3508, "nama_kab_kota": "Kab. Lumajang"},
    {"kode_kab_kota": 3509, "nama_kab_kota": "Kab. Jember"},
    {"kode_kab_kota": 3510, "nama_kab_kota": "Kab. Banyuwangi"},
    {"kode_kab_kota": 3511, "nama_kab_kota": "Kab. Bondowoso"},
    {"kode_kab_kota": 3512, "nama_kab_kota": "Kab. Situbondo"},
    {"kode_kab_kota": 3513, "nama_kab_kota": "Kab. Probolinggo"},
    {"kode_kab_kota": 3514, "nama_kab_kota": "Kab. Pasuruan"},
    {"kode_kab_kota": 3515, "nama_kab_kota": "Kab. Sidoarjo"},
    {"kode_kab_kota": 3516, "nama_kab_kota": "Kab. Mojokerto"},
    {"kode_kab_kota": 3517, "nama_kab_kota": "Kab. Jombang"},
    {"kode_kab_kota": 3518, "nama_kab_kota": "Kab. Nganjuk"},
    {"kode_kab_kota": 3519, "nama_kab_kota": "Kab. Madiun"},
    {"kode_kab_kota": 3520, "nama_kab_kota": "Kab. Magetan"},
    {"kode_kab_kota": 3521, "nama_kab_kota": "Kab. Ngawi"},
    {"kode_kab_kota": 3522, "nama_kab_kota": "Kab. Bojonegoro"},
    {"kode_kab_kota": 3523, "nama_kab_kota": "Kab. Tuban"},
    {"kode_kab_kota": 3524, "nama_kab_kota": "Kab. Lamongan"},
    {"kode_kab_kota": 3525, "nama_kab_kota": "Kab. Gresik"},
    {"kode_kab_kota": 3526, "nama_kab_kota": "Kab. Bangkalan"},
    {"kode_kab_kota": 3527, "nama_kab_kota": "Kab. Sampang"},
    {"kode_kab_kota": 3528, "nama_kab_kota": "Kab. Pamekasan"},
    {"kode_kab_kota": 3529, "nama_kab_kota": "Kab. Sumenep"},
    {"kode_kab_kota": 3571, "nama_kab_kota": "Kota Kediri"},
    {"kode_kab_kota": 3572, "nama_kab_kota": "Kota Blitar"},
    {"kode_kab_kota": 3573, "nama_kab_kota": "Kota Malang"},
    {"kode_kab_kota": 3574, "nama_kab_kota": "Kota Probolinggo"},
    {"kode_kab_kota": 3575, "nama_kab_kota": "Kota Pasuruan"},
    {"kode_kab_kota": 3576, "nama_kab_kota": "Kota Mojokerto"},
    {"kode_kab_kota": 3577, "nama_kab_kota": "Kota Madiun"},
    {"kode_kab_kota": 3578, "nama_kab_kota": "Kota Surabaya"},
    {"kode_kab_kota": 3579, "nama_kab_kota": "Kota Batu"}
]

all_data = []

# ===============================
# LEVEL 2 (KAB/KOTA)
# ===============================
for kota in kota_jatim:

    print(f"Mengambil: {kota['nama_kab_kota']} | {start_date}")

    payload = {
        "start_date": start_date,
        "end_date": end_date,
        "level": 2,
        "variant_ids": "52,51",
        "kode_provinsi": 35,
        "kode_kab_kota": kota["kode_kab_kota"],
        "pasar_id": "",
        "skip_sat_sun": "true",
        "tipe_komoditas": ""
    }

    try:
        response = requests.post(url, headers=headers, data=payload, timeout=30)

        if response.status_code == 200:
            data = response.json()

            for entry in (data.get('data') or []):
                for harga_entry in (entry.get('daftarHarga') or []):

                    df_row = pd.json_normalize(harga_entry)

                    df_row["level"] = 2
                    df_row["kota"] = kota["nama_kab_kota"]
                    df_row["kode_kab_kota"] = kota["kode_kab_kota"]
                    df_row["variant_id"] = entry["variant_id"]
                    df_row["variant"] = entry["variant"]
                    df_row["tanggal"] = df_row["date"]

                    all_data.append(df_row)

    except requests.exceptions.RequestException as e:
        print(f"Error: {e}")

    time.sleep(0.15)


# ===============================
# LEVEL 1 (PROVINSI)
# ===============================
print(f"Mengambil Jawa Timur | {start_date}")

payload_prov = {
    "start_date": start_date,
    "end_date": end_date,
    "level": 1,
    "variant_ids": "52,51",
    "kode_provinsi": 35,
    "kode_kab_kota": "",
    "pasar_id": "",
    "skip_sat_sun": "true",
    "tipe_komoditas": ""
}

try:
    response = requests.post(url, headers=headers, data=payload_prov, timeout=30)

    if response.status_code == 200:
        data = response.json()

        for entry in (data.get("data") or []):
            for harga_entry in (entry.get("daftarHarga") or []):

                df_row = pd.json_normalize(harga_entry)

                df_row["level"] = 1
                df_row["kota"] = "Jawa Timur"
                df_row["kode_kab_kota"] = 1
                df_row["variant_id"] = entry["variant_id"]
                df_row["variant"] = entry["variant"]
                df_row["tanggal"] = df_row["date"]

                all_data.append(df_row)

except requests.exceptions.RequestException as e:
    print(f"Error: {e}")


# ===============================
# GABUNG
# ===============================
if all_data:
    sp2kp_df = pd.concat(all_data, ignore_index=True)
    sp2kp_df.drop_duplicates(inplace=True)

    print("Total baris:", len(sp2kp_df))
    print(sp2kp_df.head)
else:
    print("Tidak ada data.")

Mengambil: Kab. Pacitan | 2026-02-21
Mengambil: Kab. Ponorogo | 2026-02-21
Mengambil: Kab. Trenggalek | 2026-02-21
Mengambil: Kab. Tulungagung | 2026-02-21
Mengambil: Kab. Blitar | 2026-02-21
Mengambil: Kab. Kediri | 2026-02-21
Mengambil: Kab. Malang | 2026-02-21
Mengambil: Kab. Lumajang | 2026-02-21
Mengambil: Kab. Jember | 2026-02-21
Mengambil: Kab. Banyuwangi | 2026-02-21
Mengambil: Kab. Bondowoso | 2026-02-21
Mengambil: Kab. Situbondo | 2026-02-21
Mengambil: Kab. Probolinggo | 2026-02-21
Mengambil: Kab. Pasuruan | 2026-02-21
Mengambil: Kab. Sidoarjo | 2026-02-21
Mengambil: Kab. Mojokerto | 2026-02-21
Mengambil: Kab. Jombang | 2026-02-21
Mengambil: Kab. Nganjuk | 2026-02-21
Mengambil: Kab. Madiun | 2026-02-21
Mengambil: Kab. Magetan | 2026-02-21
Mengambil: Kab. Ngawi | 2026-02-21
Mengambil: Kab. Bojonegoro | 2026-02-21
Mengambil: Kab. Tuban | 2026-02-21
Mengambil: Kab. Lamongan | 2026-02-21
Mengambil: Kab. Gresik | 2026-02-21
Mengambil: Kab. Bangkalan | 2026-02-21
Mengambil: Kab. Sa

In [64]:
sp2kp_df = clean_sp2kp(sp2kp_df)
sp2kp_df

,kode_kab_kota,tanggal,variant_id,harga,tipe_harga_id
0,3501,2026-02-23,1.0,13300,1
1,3501,2026-02-24,1.0,13300,1
2,3501,2026-02-25,1.0,13300,1
3,3501,2026-02-23,2.0,14900,1
4,3501,2026-02-24,2.0,14900,1
...,...,...,...,...,...
220,1,2026-02-24,1.0,12850,1
221,1,2026-02-25,1.0,12883,1
222,1,2026-02-23,2.0,14652,1
223,1,2026-02-24,2.0,14652,1


In [65]:
df_final = pd.concat(
    [df_all_konsumen, df_all_produsen, sp2kp_df],
    ignore_index=True
)

cols_int = ["kode_kab_kota", "variant_id", "tipe_harga_id"]

for col in cols_int:
    df_final[col] = df_final[col].astype("Int64")  # nullable integer pandas

print(df_final.dtypes)

kode_kab_kota     Int64
tanggal          object
variant_id        Int64
harga             Int64
tipe_harga_id     Int64
dtype: object


In [66]:
# Convert tanggal ke datetime
df_final["tanggal"] = pd.to_datetime(df_final["tanggal"])

# Ubah Int64 nullable → int / None
cols_int = ["kode_kab_kota", "variant_id", "tipe_harga_id", "harga"]

for col in cols_int:
    df_final[col] = df_final[col].apply(
        lambda x: int(x) if pd.notna(x) else None
    )

# Format tanggal jadi string ISO
df_final["tanggal"] = df_final["tanggal"].dt.strftime("%Y-%m-%d")
df_final

,kode_kab_kota,tanggal,variant_id,harga,tipe_harga_id
0,3501,2026-02-21,1,12467,2
1,3503,2026-02-21,1,13000,2
2,3504,2026-02-21,1,12790,2
3,3505,2026-02-21,1,12450,2
4,3507,2026-02-21,1,13000,2
...,...,...,...,...,...
756,1,2026-02-24,1,12850,1
757,1,2026-02-25,1,12883,1
758,1,2026-02-23,2,14652,1
759,1,2026-02-24,2,14652,1


In [67]:
data = df_final.to_dict(orient="records")

batch_size = 1000

for i in range(0, len(data), batch_size):
    batch = data[i:i+batch_size]
    response = supabase.table("harga_beras").insert(batch).execute()
    
    if response.data is None:
        print("Error:", response)
    else:
        print(f"Inserted {i+len(batch)} rows")

Inserted 761 rows
